In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST RESULTS FILES
# ============================================================
# Purpose:
# 1. Read multiple JSON files from the results folder
# 2. Apply an explicit schema
# 3. Combine all result files into one DataFrame
# 4. Add ingestion metadata
# 5. Write the data into a Bronze Delta table
# ============================================================


# COMMAND ----------

# Load Environment Configuration

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# Load Bronze Helper Functions

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# ============================================================
# Define Source Folder and Target Table
# ============================================================

# results is a folder containing multiple JSON files
# such as results_1950.json, results_1951.json, etc.

source_file = f"{landing_folder_path}/results"

table_name = (
    f"{catalog_name}.{schema_name}.{bronze_prefix}_results"
)

print("Source Folder :", source_file)
print("Target Table  :", table_name)


# COMMAND ----------

# ============================================================
# Step 1 - Define Schema for Results JSON Files
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    FloatType,
    DateType
)

results_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])


# COMMAND ----------

# ============================================================
# Step 2 - Read All JSON Files from Results Folder
# ============================================================
# Spark reads all JSON files from the results folder
# and combines them into one DataFrame.

results_df = (
    spark.read
    .format("json")
    .option("dateFormat", "yyyy-MM-dd")
    .schema(results_schema)
    .load(source_file)
)

display(results_df)


# COMMAND ----------

# ============================================================
# Step 3 - Check DataFrame Schema
# ============================================================

results_df.printSchema()


# COMMAND ----------

# Check total records

print("Total records:", results_df.count())


# COMMAND ----------

# ============================================================
# Step 4 - Add Ingestion Metadata
# ============================================================
# Adds:
# 1. ingestion_timestamp
# 2. source_file
#
# source_file will tell us which results JSON file
# each record originally came from.

results_final_df = add_ingestion_metadata(results_df)

display(results_final_df)


# COMMAND ----------

# ============================================================
# Step 5 - Write to Bronze Delta Table
# ============================================================

(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


# COMMAND ----------

# ============================================================
# Step 6 - Validate Bronze Delta Table
# ============================================================

display(
    spark.table(table_name)
)


# COMMAND ----------

# Final record count validation

print(
    "Total Bronze records:",
    spark.table(table_name).count()
)

print(
    "Bronze table created successfully:",
    table_name
)